# Pairwise pruning comparison: VGG-16 and DenseNet-121

This Colab notebook runs the same full-network experiment used for the ResNet-32 benchmark, but only for VGG-16 and DenseNet-121. ResNet-32 is intentionally excluded.

Compared methods: additive RDP, pair-corrected RDP, direct RDP, and reconstructed RDHP. The default protocol uses seeds 42/43/44, 1,024 calibration examples, 1,000 test examples, 50% and 75% target sparsity, 20 curve points, 24 measured layer pairs, and 5/20 fine-tuning epochs.

The VGG-16 and DenseNet-121 definitions are explicit CIFAR-compatible adaptations in `cifar_model_zoo.py`; they should not be described as exact author-code reproductions.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys

from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
EXPERIMENT_REPO = Path('/content/rdhp-rdp-colab-experiment')
THESIS_REPO = Path('/content/my-thesis')

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('GitHub token with read access: ')
if not token:
    raise RuntimeError('A GitHub token is required to clone the private repositories.')

askpass = Path('/tmp/github-askpass-cross-model.sh')
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    "  *Username*) printf '%s\\n' x-access-token ;;\n"
    "  *Password*) printf '%s\\n' \"$GITHUB_TOKEN_FOR_ASKPASS\" ;;\n"
    "  *) printf '\\n' ;;\n"
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GITHUB_TOKEN_FOR_ASKPASS': token,
})

def clone_or_update(url, destination):
    if not (destination / '.git').exists():
        subprocess.run(['git', 'clone', url, str(destination)], env=git_env, check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], env=git_env, check=True)

try:
    clone_or_update('https://github.com/hamidrg20001379/my-thesis.git', THESIS_REPO)
    clone_or_update('https://github.com/hamidrg20001379/rdhp-rdp-colab-experiment.git', EXPERIMENT_REPO)
finally:
    askpass.unlink(missing_ok=True)

RUNNER = EXPERIMENT_REPO / 'run_cross_model_pairwise.py'
assert RUNNER.exists(), RUNNER
print('Runner:', RUNNER)
print('Thesis checkout:', THESIS_REPO)

In [ ]:
MODELS = ['vgg16', 'densenet121']
SEEDS = [42, 43, 44]
TARGET_SPARSITIES = [0.50, 0.75]
TRAIN_SAMPLES = 5000
CALIB_SAMPLES = 1024
TEST_SAMPLES = 1000
BATCH_SIZE = 128
CURVE_POINTS = 20
BUDGET_BINS = 5000
PAIR_COUNT = 24
PAIR_REFERENCE_RATIO = 0.50
FINETUNE_EPOCHS = [5, 20]
LEARNING_RATE = 0.1

DATA_ROOT = DRIVE_ROOT / 'cifar10'
DATASET_DIR = DATA_ROOT / 'cifar-10-batches-py'
required_files = ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch']
missing_data = [name for name in required_files if not (DATASET_DIR / name).exists()]
if missing_data:
    raise FileNotFoundError(f'Missing CIFAR-10 files under {DATASET_DIR}: {missing_data}')

CHECKPOINT_PATTERN = DRIVE_ROOT / 'cross_model_checkpoints/{model}/seed_{seed}/dense.pt'
checkpoint_paths = [Path(str(CHECKPOINT_PATTERN).format(model=model, seed=seed)) for model in MODELS for seed in SEEDS]
missing_checkpoints = [path for path in checkpoint_paths if not path.exists()]

# Keep this at 0 when using prepared dense checkpoints. If checkpoints are
# unavailable, set it to 20 and the runner will train each dense model.
TRAIN_DENSE_EPOCHS = 0

print('Models:', MODELS)
print('Seeds:', SEEDS)
print('CIFAR-10:', DATASET_DIR)
print('Missing dense checkpoints:', len(missing_checkpoints))
for path in missing_checkpoints[:6]:
    print('  ', path)
if missing_checkpoints and TRAIN_DENSE_EPOCHS <= 0:
    print('Set TRAIN_DENSE_EPOCHS=20 before running, or upload the expected checkpoints.')

## Structural validation

This checks that both CIFAR-compatible model factories produce 10-class outputs and expose prunable layers. It does not use CIFAR-10 accuracy.

In [ ]:
subprocess.run([sys.executable, str(RUNNER), '--self-test'], check=True)

## Run the real experiments

Set `RUN_REAL = True` only after the CIFAR-10 files and dense checkpoints are ready. Use a new output directory for each run; the runner refuses to overwrite previous results.

In [ ]:
RUN_REAL = False
OUTPUT_ROOT = DRIVE_ROOT / 'rdhp_rdp_vgg16_densenet121_pairwise_all_seeds'

if not RUN_REAL:
    print('RUN_REAL is False; no research experiment was started.')
else:
    if OUTPUT_ROOT.exists():
        raise FileExistsError(f'Choose a new output directory: {OUTPUT_ROOT}')
    command = [
        sys.executable, str(RUNNER),
        '--model', *MODELS,
        '--thesis-root', str(THESIS_REPO),
        '--data-root', str(DATA_ROOT),
        '--train-samples', str(TRAIN_SAMPLES),
        '--calib-samples', str(CALIB_SAMPLES),
        '--test-samples', str(TEST_SAMPLES),
        '--batch-size', str(BATCH_SIZE),
        '--target-sparsities', *(str(value) for value in TARGET_SPARSITIES),
        '--curve-points', str(CURVE_POINTS),
        '--budget-bins', str(BUDGET_BINS),
        '--pair-count', str(PAIR_COUNT),
        '--pair-reference-ratio', str(PAIR_REFERENCE_RATIO),
        '--finetune-epochs', *(str(value) for value in FINETUNE_EPOCHS),
        '--learning-rate', str(LEARNING_RATE),
        '--seeds', *(str(seed) for seed in SEEDS),
        '--output', str(OUTPUT_ROOT),
    ]
    if missing_checkpoints:
        if TRAIN_DENSE_EPOCHS <= 0:
            raise RuntimeError('Dense checkpoints are missing and TRAIN_DENSE_EPOCHS is not positive.')
        command += ['--train-epochs', str(TRAIN_DENSE_EPOCHS)]
    else:
        command += ['--checkpoint-pattern', str(CHECKPOINT_PATTERN)]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
    print('Completed:', OUTPUT_ROOT)

## Accuracy summary

This cell reads only completed runner outputs. The synthetic smoke test and untrained/random runs must not be used as thesis accuracy evidence.

In [ ]:
import pandas as pd

results_file = OUTPUT_ROOT / 'all_models_method_results.csv'
if not results_file.exists():
    print('No completed real run found:', results_file)
else:
    results = pd.read_csv(results_file)
    summary = (results.groupby(['model', 'target_sparsity', 'method'], as_index=False)
               .agg(seeds=('seed', 'nunique'),
                    dense_accuracy=('dense_accuracy', 'mean'),
                    immediate_accuracy=('immediate_accuracy', 'mean'),
                    finetuned_accuracy_5=('finetuned_accuracy_5', 'mean'),
                    finetuned_accuracy_20=('finetuned_accuracy_20', 'mean'),
                    actual_sparsity=('sparsity', 'mean'),
                    flops_reduction=('flops_reduction_fraction', 'mean')))
    summary['delta_dense_5'] = summary['finetuned_accuracy_5'] - summary['dense_accuracy']
    summary['delta_dense_20'] = summary['finetuned_accuracy_20'] - summary['dense_accuracy']
    display(summary)
    summary.to_csv(OUTPUT_ROOT / 'accuracy_summary.csv', index=False)
    print('Saved:', OUTPUT_ROOT / 'accuracy_summary.csv')

## Interpretation guardrails

Report the results as a bounded comparison of the stated CIFAR-compatible adaptations. Keep pair-corrected RDP identified as a heuristic, report achieved sparsity rather than only target sparsity, and do not convert approximate FLOPs into measured latency or speedup.